<a href="https://colab.research.google.com/github/RENSHUZHE/HKU-DQMC/blob/main/DQMC_Theory_4_Checkerboard_decomposition_and_self_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Checkerboard Decomposition


For certain matrices on the exponent of e, such as the nearest-neighbor hopping matrix mentioned earlier, or matrices of auxiliary fields in similar forms, we usually divide them into different families. Each family consists of mutually commuting (block) small matrices. This utilizes the sparse nature of the matrices, facilitating the calculation of the matrix $\exp$ and matrix multiplication.

Taking the nearest-neighbor hopping on a $4 \times 4$ lattice as an example:

$$\begin{equation}
\begin{aligned}
e^{-\Delta \tau \hat{T}} &=\prod_{\sigma} e^{-\Delta \tau t \sum_{\langle i j\rangle}\left(\hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+\hat{c}_{j \sigma}^{\dagger} \hat{c}_{i \sigma}\right)} \\
&=\prod_{\sigma} e^{-\Delta \tau t\left[\sum_{\langle i j\rangle \in \operatorname{fam}_{1}}\left(\hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+\hat{c}_{j \sigma}^{\dagger} \hat{c}_{i \sigma}\right)+\sum_{\langle i j\rangle \in \mathrm{fam}_{2}}\left(\hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+\hat{c}_{j \sigma}^{\dagger} \hat{c}_{i \sigma}\right)\right]} \\
& \approx \prod_{\sigma} e^{-\Delta \tau t\left[\sum_{\langle i j\rangle \in \operatorname{fam}_{1}}\left(\hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+\hat{c}_{j \sigma}^{\dagger} \hat{c}_{i \sigma}\right)\right]}e^{-\Delta \tau t\left[\sum_{\langle i j\rangle \in \mathrm{fam}_{2}}\left(\hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+\hat{c}_{j \sigma}^{\dagger} \hat{c}_{i \sigma}\right)\right]} \\
&=\prod_{\sigma} \prod_{m=1}^{2} \prod_{n=1}^{N / 4} e^{-\Delta \tau (-t)\sum_{\langle i j\rangle \in \square_{m n}}\left(\hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+\hat{c}_{j \sigma}^{\dagger} \hat{c}_{i \sigma}\right)}
\end{aligned}
\end{equation}$$

As shown in the figure below, the two families are red and blue, respectively. It can be seen that, for example, different red squares do not share common sites, meaning they commute with each other (block-diagonal), and each square is a matrix of the form:

$$\begin{equation}
\begin{pmatrix}
0 & -t & 0 &-t \\
-t &0 & -t & 0\\
0& -t & 0 & -t\\
-t & 0 & -t & 0
\end{pmatrix} \end{equation}$$

We only need to diagonalize this matrix to calculate its $\exp$, obtaining $4 \times 4$ matrices. Also, when performing matrix multiplication later, we can multiply these different $4 \times 4$ matrices separately with the block partitions of the large matrix.

![Checkerboard decomposition example](https://quantummc.xyz/wp-content/uploads/2023/03/WechatIMG1841-300x300.jpeg)

&nbsp;


### Self-learning Monte Carlo



We know that for a specific boson configuration, we can calculate its corresponding weight $W$. And for a classical Hamiltonian containing only bosons, we have $W=e^{-\beta H}$.

In the presence of fermions, our weight is related to a certain determinant, and undoubtedly calculating the determinant is a very time-consuming task. A natural thought is that since we are sampling the configuration space of the auxiliary fields (bosons) during our sampling process (the fermion parts are traced out respectively), if we can find an effective Hamiltonian containing only bosons, we would only need to calculate the energy difference of the bosons before and after. Namely, what we expect is:

$$\begin{equation}
W=e^{ -\beta H[\mathcal{C}]}=\operatorname{Det}\left[\mathbf{I}+ \mathbf{B}_{C}^{\sigma}(\beta, 0)\right] \longrightarrow W=e^{-\beta H^{\mathrm{eff}}}
\end{equation}$$

However, strict equivalence is very difficult to achieve, but we can approximately find an effective Hamiltonian $H^{\mathrm{eff}}$, such that $\qquad \operatorname{Det}\left[\mathbf{I}+ \mathbf{B}_{C}^{\sigma}(\beta, 0)\right] \approx e^{-\beta H_{eff}}$.

For example, we can assume the form of the effective Hamiltonian to be a nearest-neighbor Ising model with parameter $J_1$, or an Ising model that also includes next-nearest-neighbor $J_2$, next-next-nearest-neighbor $J_3$, etc. And the values of these parameters can be learned through the weights corresponding to different configurations. This process can utilize various machine learning methods.

For a certain configuration $x_i$, the corresponding weight is $W_i$. We adjust the values of the various parameters $J$ such that:
$W_i \approx e^{-\beta H^{\mathrm{eff}}(x_i)}$

Once we have the effective Hamiltonian, we can use the weights corresponding to the effective Hamiltonian for updates. However, such updates will have some differences from the actual updates, so after performing Monte Carlo updates for several steps using the weights corresponding to the effective Hamiltonian, we finally need to calculate the weight ratio between the initial and updated states according to the original Hamiltonian, adding an extra acceptance probability to correct the total acceptance probability so that it satisfies the detailed balance condition. This acceptance probability is:

$$\begin{equation}
A\left(\mathcal{C} \rightarrow \mathcal{C}^{\prime}\right)=\min \left\{1, \frac{\exp \left(-\beta H\left[\mathcal{C}^{\prime}\right]\right)}{\exp (-\beta H[\mathcal{C}])} \frac{\exp \left(-\beta H^{\mathrm{eff}}[\mathcal{C}]\right)}{\exp \left(-\beta H^{\mathrm{eff}}\left[\mathcal{C}^{\prime}\right]\right)}\right\}
\end{equation}$$

By the way, the usual experience is that an effective model learned at a certain temperature and size (for instance, having learned the value of $J_1$) can be applied to a slightly larger size at the same temperature.